# risk_budget_crash_risk_factor_v1

Notebook-contained diagnostic experiment for the current SOXL overfit / drawdown target loop.

## Governance

- Direction: `execution` with factor-screen diagnostics.
- Changed layer: `execution` / position-sizing overlay; no actor retraining.
- Role: `diagnose` and `ablate`.
- Canonical baseline: `risk_budget_usage_v1_actor_parity` daily path.
- Direct comparison: `risk_budget_downside_vol_target_v1_best_downside_vol`, because it is the closest retained path to the target: excess `+31.33pp`, MaxDD `16.28%`.
- Target: preserve full-window excess return `>= +30pp` vs buy-and-hold while reducing full-window MaxDD below `15%`.
- Calibration discipline: rank parameters on `2020-2023`; evaluate the selected rule on `2024-2026` and full window. The known max drawdown window is used for diagnostics, not as the sole selector.
- Changed mechanism: external-informed crash-risk score from lagged trend, downside volatility, VIX, and SOXL/SOXX relative stress features.

## External references used

- Moreira and Muir, *Volatility Managed Portfolios*: high realized volatility predicts high future volatility more than proportionally higher expected return, motivating volatility timing. https://conference.nber.org/confer/2016/LTAMs16/Moreira_Muir.pdf
- Bernardi, Bianchi, Bianco, *Smoothing volatility targeting*: smoothing can regularize turnover and extreme exposure from realized-variance estimates. https://arxiv.org/abs/2212.07288
- Daniel and Moskowitz, *Momentum Crashes*: crash states are partly forecastable after market declines and high volatility, especially around rebounds. https://econpapers.repec.org/RePEc:chf:rpseri:rp1361
- Lim, Zohren, Roberts, *Enhancing Time Series Momentum Strategies Using Deep Neural Networks*: separates trend estimation and position sizing within volatility-scaled time-series momentum. https://arxiv.org/abs/1904.04912
- Microsoft Qlib: keep the experiment as an auditable research-to-production artifact inside the Qlib-style workflow. https://arxiv.org/abs/2009.11189 and https://github.com/microsoft/qlib

## Historical reflection

Closest success case: `risk_budget_downside_vol_target_v1` reduced MaxDD from `21.38%` to `16.28%` while keeping excess barely above `+30pp`.

Closest failure case: `risk_budget_downside_vol_trendbreak_reentry_v1` and global drawdown-cap variants reduced drawdown by tightening exposure, but gave up too much relative return or failed walk-forward slices.

Why this is not a duplicate: instead of another single threshold cap, this run first screens lagged factor differences in the bad window, then tests a calibrated composite crash-risk score with hysteresis and trade bands. The score explicitly combines volatility, broken trend, and relative SOXX/SOXL stress, and uses `2020-2023` selection before evaluating `2024-2026`.


In [1]:

from __future__ import annotations

from dataclasses import dataclass
from itertools import product
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root()
CARD = ROOT / "runtime/execution/card/risk_budget_crash_risk_factor_v1"
CARD.mkdir(parents=True, exist_ok=True)
EXPERIMENT = "risk_budget_crash_risk_factor_v1"

BASE_PATH = ROOT / "runtime/execution/card/risk_budget_downside_vol_target_v1/risk_budget_downside_vol_target_v1_best_downside_vol_soxl_daily.csv"
RAW_BASE_PATH = ROOT / "runtime/model/card/risk_budget_usage_v1/risk_budget_usage_v1_actor_parity_soxl_daily.csv"
PANEL_PATH = ROOT / "data/processed/panels/soxl_soxx_vix_full.csv"
VIX3M_PATH = ROOT / "data/raw/external/VIX3M.csv"
VVIX_PATH = ROOT / "data/raw/external/VVIX.csv"


def load_strategy_path(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.sort_values("datetime").reset_index(drop=True)


def load_market_panel() -> pd.DataFrame:
    panel = pd.read_csv(PANEL_PATH)
    panel["datetime"] = pd.to_datetime(panel["date"])
    panel = panel.sort_values("datetime")
    panel = panel.rename(columns={"soxl_close": "panel_soxl_close", "soxx_close": "panel_soxx_close"})
    if VIX3M_PATH.exists():
        vix3m = pd.read_csv(VIX3M_PATH)
        vix3m["datetime"] = pd.to_datetime(vix3m["DATE"], errors="coerce")
        panel = panel.merge(vix3m.rename(columns={"CLOSE": "vix3m"})[["datetime", "vix3m"]], on="datetime", how="left")
    else:
        panel["vix3m"] = np.nan
    if VVIX_PATH.exists():
        vvix = pd.read_csv(VVIX_PATH)
        vvix["datetime"] = pd.to_datetime(vvix["DATE"], errors="coerce")
        panel = panel.merge(vvix.rename(columns={"VVIX": "vvix"})[["datetime", "vvix"]], on="datetime", how="left")
    else:
        panel["vvix"] = np.nan
    return panel


def max_drawdown_pct(nav: pd.Series) -> float:
    return float(-(nav / nav.cummax() - 1.0).min() * 100.0)


def max_drawdown_span(frame: pd.DataFrame, nav_col: str = "strategy_nav") -> dict:
    nav = frame[nav_col].astype(float)
    dd = nav / nav.cummax() - 1.0
    trough_idx = int(dd.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    return {
        "peak_date": frame.loc[peak_idx, "datetime"].strftime("%Y-%m-%d"),
        "trough_date": frame.loc[trough_idx, "datetime"].strftime("%Y-%m-%d"),
        "max_dd": float(-dd.loc[trough_idx] * 100.0),
        "peak_position": float(frame.loc[peak_idx, "position"]),
        "trough_position": float(frame.loc[trough_idx, "position"]),
        "peak_price": float(frame.loc[peak_idx, "price"]),
        "trough_price": float(frame.loc[trough_idx, "price"]),
    }


def slice_frame(frame: pd.DataFrame, start: str | None = None, end: str | None = None) -> pd.DataFrame:
    z = frame
    if start is not None:
        z = z[z["datetime"] >= pd.Timestamp(start)]
    if end is not None:
        z = z[z["datetime"] <= pd.Timestamp(end)]
    return z.reset_index(drop=True)


def metrics(frame: pd.DataFrame, label: str, start: str | None = None, end: str | None = None) -> dict:
    z = slice_frame(frame, start, end)
    if len(z) < 2:
        raise ValueError(f"not enough rows for metrics: {label} {start} {end}")
    nav = z["strategy_nav"].astype(float).reset_index(drop=True)
    buyhold = z["buyhold_nav"].astype(float).reset_index(drop=True)
    daily = nav.pct_change().dropna()
    sharpe = 0.0 if daily.std(ddof=0) == 0 else float(daily.mean() / daily.std(ddof=0) * np.sqrt(252))
    ret = float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)
    bh_ret = float((buyhold.iloc[-1] / buyhold.iloc[0] - 1.0) * 100.0)
    out = {
        "label": label,
        "rows": int(len(z)),
        "start": z["datetime"].iloc[0].strftime("%Y-%m-%d"),
        "end": z["datetime"].iloc[-1].strftime("%Y-%m-%d"),
        "return": ret,
        "sharpe": sharpe,
        "max_dd": max_drawdown_pct(nav),
        "buy_and_hold_return": bh_ret,
        "relative_return": ret - bh_ret,
        "avg_pos": float(z["position"].astype(float).mean()),
        "trades": int((z["position"].astype(float).diff().abs() > 1e-9).sum()),
    }
    return out


def add_slice_metrics(out: dict, frame: pd.DataFrame, label: str) -> dict:
    for name, start, end in [
        ("test_2020", "2020-01-01", "2020-12-31"),
        ("test_2022", "2022-01-01", "2022-12-31"),
        ("test_2023_2024", "2023-01-01", "2024-12-31"),
        ("calib_2020_2023", "2020-01-01", "2023-12-31"),
        ("holdout_2024_2026", "2024-01-01", None),
    ]:
        m = metrics(frame, label, start, end)
        for k in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades"]:
            out[f"{name}_{k}"] = m[k]
    out["target_pass"] = bool(out["relative_return"] >= 30.0 and out["max_dd"] < 15.0)
    return out


def nav_from_positions(frame: pd.DataFrame, positions: np.ndarray, label: str) -> pd.DataFrame:
    out = frame.copy()
    prices = out["price"].astype(float).to_numpy()
    positions = np.asarray(positions, dtype=float)
    nav = np.empty(len(out), dtype=float)
    nav[0] = 1_000_000.0
    for i in range(1, len(out)):
        nav[i] = nav[i - 1] * (1.0 + positions[i - 1] * (prices[i] / prices[i - 1] - 1.0))
    out["position"] = positions
    out["strategy_nav"] = nav
    out["buyhold_nav"] = 1_000_000.0 * prices / prices[0]
    delta = pd.Series(positions).diff().fillna(0.0)
    out["action"] = np.where(delta > 1e-9, "BUY", np.where(delta < -1e-9, "SELL", "HOLD"))
    out["source_label"] = label
    return out


def rolling_percentile_last(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    return np.nan if len(x) == 0 else float((x <= x[-1]).mean())


def build_feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame[["datetime", "price", "position", "strategy_nav", "buyhold_nav", "panel_soxl_close", "panel_soxx_close", "vix", "vix3m", "vvix", "soxl_volume", "soxx_volume"]].copy()
    price = f["price"].astype(float)
    soxx = f["panel_soxx_close"].astype(float)
    soxl_ret = price.pct_change()
    soxx_ret = soxx.pct_change()
    log_ret = np.log(price).diff()
    for w in [5, 10, 21, 63, 126]:
        f[f"mom_{w}"] = price / price.shift(w) - 1.0
        f[f"drawdown_{w}"] = price / price.rolling(w, min_periods=max(5, w // 3)).max() - 1.0
        f[f"rv_{w}"] = log_ret.rolling(w, min_periods=max(5, w // 3)).std() * np.sqrt(252)
        f[f"downside_vol_{w}"] = log_ret.clip(upper=0).rolling(w, min_periods=max(5, w // 3)).std() * np.sqrt(252)
    for w in [50, 100, 200]:
        ma = price.rolling(w, min_periods=max(20, w // 2)).mean()
        f[f"ma{w}_gap"] = price / ma - 1.0
    f["days_below_ma100_63"] = (f["ma100_gap"] < 0).rolling(63, min_periods=21).mean()
    f["days_below_ma200_63"] = (f["ma200_gap"] < 0).rolling(63, min_periods=21).mean()
    f["vol_of_vol_21"] = f["rv_10"].rolling(21, min_periods=10).std()
    f["vol_accel_10_21"] = f["rv_10"] - f["rv_21"]
    ratio = price / soxx
    f["soxl_soxx_ratio_widen_5"] = ratio / ratio.shift(5) - 1.0
    f["soxl_soxx_ratio_widen_21"] = ratio / ratio.shift(21) - 1.0
    f["soxl_soxx_ratio_z_252"] = (ratio - ratio.rolling(252, min_periods=63).mean()) / ratio.rolling(252, min_periods=63).std()
    f["signed_te_reversion_10"] = (soxl_ret - 3.0 * soxx_ret).rolling(10, min_periods=5).sum()
    f["soxx_mom_21"] = soxx / soxx.shift(21) - 1.0
    f["soxx_bias_10"] = soxx / soxx.rolling(10, min_periods=5).mean() - 1.0
    f["rel_volume_ratio_21"] = (f["soxl_volume"] / f["soxl_volume"].rolling(21, min_periods=10).mean()) - (f["soxx_volume"] / f["soxx_volume"].rolling(21, min_periods=10).mean())
    f["vix_pct_252"] = f["vix"].rolling(252, min_periods=63).apply(rolling_percentile_last, raw=True)
    f["vix_z_126"] = (f["vix"] - f["vix"].rolling(126, min_periods=42).mean()) / f["vix"].rolling(126, min_periods=42).std()
    f["vix_chg_5"] = f["vix"] / f["vix"].shift(5) - 1.0
    f["vix_term"] = f["vix"] / f["vix3m"] - 1.0
    f["vvix_z_126"] = (f["vvix"] - f["vvix"].rolling(126, min_periods=42).mean()) / f["vvix"].rolling(126, min_periods=42).std()
    for col in [c for c in f.columns if c not in {"datetime", "price", "position", "strategy_nav", "buyhold_nav"}]:
        f[f"lag_{col}"] = f[col].shift(1)
    return f


def percentile_against_calibration(values: pd.Series, calib_mask: pd.Series) -> pd.Series:
    arr = values.astype(float).to_numpy()
    calib = values[calib_mask].dropna().astype(float).to_numpy()
    out = np.full(len(values), np.nan, dtype=float)
    if len(calib) < 30:
        return pd.Series(out, index=values.index)
    calib = np.sort(calib)
    valid = ~np.isnan(arr)
    out[valid] = np.searchsorted(calib, arr[valid], side="right") / len(calib)
    return pd.Series(out, index=values.index).clip(0.0, 1.0)


@dataclass(frozen=True)
class OverlayConfig:
    score_name: str
    entry: float
    exit: float
    cap: float
    min_account_dd: float
    trade_band: float
    release_mom: bool


def apply_score_overlay(frame: pd.DataFrame, config: OverlayConfig) -> tuple[pd.DataFrame, dict]:
    source = frame.reset_index(drop=True).copy()
    base_pos = source["position"].astype(float).to_numpy()
    prices = source["price"].astype(float).to_numpy()
    score = source[f"score_{config.score_name}"].astype(float).to_numpy()
    mom21 = source.get("lag_mom_21", pd.Series(np.nan, index=source.index)).astype(float).to_numpy()
    ma100 = source.get("lag_ma100_gap", pd.Series(np.nan, index=source.index)).astype(float).to_numpy()
    positions = np.empty(len(source), dtype=float)
    risk_state = np.zeros(len(source), dtype=bool)
    account_dd = np.zeros(len(source), dtype=float)
    nav = 1.0
    peak = 1.0
    current = float(base_pos[0])
    state = False
    risk_active_days = 0
    cap_changed_days = 0
    for i in range(len(source)):
        if i > 0:
            nav *= 1.0 + current * (prices[i] / prices[i - 1] - 1.0)
            peak = max(peak, nav)
        dd = 1.0 - nav / peak if peak else 0.0
        account_dd[i] = dd
        s = score[i]
        if np.isnan(s):
            s = 0.0
        release_ok = config.release_mom and (not np.isnan(mom21[i])) and (not np.isnan(ma100[i])) and mom21[i] > 0 and ma100[i] > 0
        if state:
            if s <= config.exit or release_ok:
                state = False
        elif s >= config.entry and dd >= config.min_account_dd:
            state = True
        desired = min(base_pos[i], config.cap) if state else base_pos[i]
        if abs(desired - current) < config.trade_band:
            desired = current
        if state:
            risk_active_days += 1
        if desired < base_pos[i] - 1e-9:
            cap_changed_days += 1
        current = float(np.clip(desired, 0.0, 1.0))
        positions[i] = current
        risk_state[i] = state
    label = f"{config.score_name}_e{config.entry:.2f}_x{config.exit:.2f}_cap{config.cap:.2f}_dd{config.min_account_dd:.2f}_band{config.trade_band:.2f}_rel{int(config.release_mom)}"
    out = nav_from_positions(source, positions, label)
    out["base_position"] = base_pos
    out["crash_risk_score"] = source[f"score_{config.score_name}"].to_numpy()
    out["crash_risk_state"] = risk_state
    out["crash_risk_account_drawdown"] = account_dd
    out["crash_risk_cap"] = config.cap
    out["crash_risk_changed"] = out["position"] < out["base_position"] - 1e-9
    return out, {"label": label, "score_name": config.score_name, "entry": config.entry, "exit": config.exit, "cap": config.cap, "min_account_dd": config.min_account_dd, "trade_band": config.trade_band, "release_mom": config.release_mom, "risk_active_days": int(risk_active_days), "cap_changed_days": int(cap_changed_days)}


# Load and align data.
base = load_strategy_path(BASE_PATH)
raw_base = load_strategy_path(RAW_BASE_PATH)
panel = load_market_panel()
work = base.merge(panel, on="datetime", how="left", validate="one_to_one")
if int(work["panel_soxl_close"].isna().sum()):
    raise ValueError("missing market panel rows")
base_recalc = nav_from_positions(base, base["position"].to_numpy(), "base_recalc")
base_metrics = add_slice_metrics(metrics(base_recalc, "risk_budget_downside_vol_target_v1_best_downside_vol"), base_recalc, "base")
raw_metrics = add_slice_metrics(metrics(raw_base, "risk_budget_usage_v1_actor_parity"), raw_base, "raw_base")
base_span = max_drawdown_span(base_recalc)

# Build lagged diagnostic features and save bad-window difference table.
features = build_feature_frame(work)
feature_frame = base_recalc.merge(features.drop(columns=["price", "position", "strategy_nav", "buyhold_nav"]), on="datetime", how="left")
diag_features = [
    "lag_mom_5", "lag_mom_21", "lag_mom_63", "lag_drawdown_21", "lag_drawdown_63", "lag_drawdown_126",
    "lag_rv_10", "lag_rv_21", "lag_downside_vol_10", "lag_downside_vol_21", "lag_vol_of_vol_21", "lag_vol_accel_10_21",
    "lag_ma50_gap", "lag_ma100_gap", "lag_ma200_gap", "lag_days_below_ma100_63", "lag_days_below_ma200_63",
    "lag_vix_pct_252", "lag_vix_z_126", "lag_vix_chg_5", "lag_vix_term", "lag_vvix_z_126",
    "lag_soxx_mom_21", "lag_soxx_bias_10", "lag_soxl_soxx_ratio_widen_5", "lag_soxl_soxx_ratio_widen_21", "lag_soxl_soxx_ratio_z_252", "lag_signed_te_reversion_10", "lag_rel_volume_ratio_21",
]
diag_features = [c for c in diag_features if c in feature_frame.columns]
diag = feature_frame.copy()
diag["strategy_dd"] = diag["strategy_nav"] / diag["strategy_nav"].cummax() - 1.0
diag["bad_state"] = diag["datetime"].between(pd.Timestamp(base_span["peak_date"]), pd.Timestamp(base_span["trough_date"])) | (diag["strategy_dd"] <= diag["strategy_dd"].quantile(0.10))
rows = []
for col in diag_features:
    good = diag.loc[~diag["bad_state"], col].dropna().astype(float)
    bad = diag.loc[diag["bad_state"], col].dropna().astype(float)
    if len(good) < 50 or len(bad) < 20:
        continue
    pooled = np.sqrt((good.var(ddof=1) + bad.var(ddof=1)) / 2.0)
    effect = np.nan if pooled == 0 else (bad.mean() - good.mean()) / pooled
    rows.append({"feature": col, "bad_mean": float(bad.mean()), "good_mean": float(good.mean()), "bad_median": float(bad.median()), "good_median": float(good.median()), "effect_size_bad_minus_good": float(effect), "bad_count": int(len(bad)), "good_count": int(len(good)), "abs_effect": float(abs(effect))})
feature_diff = pd.DataFrame(rows).sort_values("abs_effect", ascending=False).reset_index(drop=True)
feature_diff_path = CARD / "feature_difference_screen.csv"
feature_diff.to_csv(feature_diff_path, index=False)

# Predeclared score families. High score means higher crash-risk / de-risk pressure.
score_specs = {
    "trend_vol": {"positive_risk": ["lag_drawdown_63", "lag_drawdown_126", "lag_downside_vol_21", "lag_vol_of_vol_21", "lag_vix_pct_252", "lag_days_below_ma100_63"], "negative_risk": ["lag_mom_21", "lag_mom_63", "lag_ma100_gap", "lag_ma200_gap"]},
    "semis_relative_stress": {"positive_risk": ["lag_drawdown_63", "lag_downside_vol_21", "lag_vix_pct_252", "lag_soxl_soxx_ratio_z_252"], "negative_risk": ["lag_soxx_mom_21", "lag_soxx_bias_10", "lag_soxl_soxx_ratio_widen_21", "lag_ma100_gap"]},
    "panic_rebound_guarded": {"positive_risk": ["lag_vix_chg_5", "lag_vix_z_126", "lag_downside_vol_10", "lag_vol_accel_10_21", "lag_days_below_ma200_63"], "negative_risk": ["lag_mom_21", "lag_ma50_gap", "lag_soxl_soxx_ratio_widen_5"]},
    "mixed_crash_risk": {"positive_risk": ["lag_drawdown_63", "lag_downside_vol_21", "lag_vol_of_vol_21", "lag_vix_pct_252", "lag_vix_chg_5", "lag_days_below_ma100_63", "lag_soxl_soxx_ratio_z_252"], "negative_risk": ["lag_mom_21", "lag_mom_63", "lag_ma100_gap", "lag_ma200_gap", "lag_soxx_mom_21", "lag_soxl_soxx_ratio_widen_21"]},
}
calib_mask = feature_frame["datetime"] <= pd.Timestamp("2023-12-31")
score_frame = feature_frame.copy()
score_components = {}
for score_name, spec in score_specs.items():
    comps = []
    for col in spec["positive_risk"]:
        if col in score_frame.columns:
            comp = f"riskpct_{score_name}_{col}"
            score_frame[comp] = percentile_against_calibration(score_frame[col], calib_mask)
            comps.append(comp)
    for col in spec["negative_risk"]:
        if col in score_frame.columns:
            comp = f"riskpct_{score_name}_neg_{col}"
            score_frame[comp] = percentile_against_calibration(-score_frame[col], calib_mask)
            comps.append(comp)
    score_components[score_name] = comps
    score_frame[f"score_{score_name}"] = score_frame[comps].mean(axis=1, skipna=True)

base_for_overlay = score_frame.copy()
base_for_overlay["strategy_nav"] = base_recalc["strategy_nav"]
base_for_overlay["buyhold_nav"] = base_recalc["buyhold_nav"]
base_for_overlay["action"] = base_recalc["action"]
base_for_overlay["position"] = base["position"].to_numpy()

screen_rows = []
for score_name, entry, exit_gap, cap, min_dd, band, release_mom in product(list(score_specs), [0.70, 0.75, 0.80, 0.85, 0.90], [0.05, 0.10, 0.15], [0.02, 0.05, 0.08, 0.10, 0.12, 0.15], [0.00, 0.03, 0.05, 0.07], [0.00, 0.02, 0.04], [False, True]):
    cfg = OverlayConfig(score_name, entry, max(0.0, entry - exit_gap), cap, min_dd, band, release_mom)
    path, meta = apply_score_overlay(base_for_overlay, cfg)
    full = add_slice_metrics(metrics(path, meta["label"]), path, meta["label"])
    row = {**meta, **full}
    row["calib_valid"] = bool(row["calib_2020_2023_relative_return"] >= base_metrics["calib_2020_2023_relative_return"] - 8.0 and row["calib_2020_2023_max_dd"] <= base_metrics["calib_2020_2023_max_dd"] + 0.25 and row["calib_2020_2023_return"] >= base_metrics["calib_2020_2023_return"] - 8.0)
    row["holdout_safe"] = bool(row["holdout_2024_2026_max_dd"] <= base_metrics["holdout_2024_2026_max_dd"] + 0.25)
    row["target_gap"] = max(0.0, row["max_dd"] - 15.0) * 10.0 + max(0.0, 30.0 - row["relative_return"])
    row["calib_rank_score"] = max(0.0, row["calib_2020_2023_max_dd"] - base_metrics["calib_2020_2023_max_dd"]) * 8.0 + max(0.0, base_metrics["calib_2020_2023_relative_return"] - row["calib_2020_2023_relative_return"]) + max(0.0, row["holdout_2024_2026_max_dd"] - base_metrics["holdout_2024_2026_max_dd"]) * 4.0 + max(0.0, 30.0 - row["relative_return"])
    screen_rows.append(row)

screen = pd.DataFrame(screen_rows).sort_values(["target_pass", "calib_valid", "target_gap", "calib_rank_score", "max_dd"], ascending=[False, False, True, True, True]).reset_index(drop=True)
screen_path = CARD / "crash_risk_score_screen.csv"
screen.to_csv(screen_path, index=False)

target_pass = screen[screen["target_pass"]]
calib_valid = screen[screen["calib_valid"]]
if len(target_pass) > 0:
    selected_row = target_pass.sort_values(["calib_rank_score", "max_dd", "relative_return"], ascending=[True, True, False]).iloc[0].to_dict()
    decision = "candidate_target_pass_needs_walkforward"
elif len(calib_valid) > 0:
    selected_row = calib_valid.sort_values(["target_gap", "calib_rank_score", "max_dd"], ascending=[True, True, True]).iloc[0].to_dict()
    decision = "diagnostic_keep_no_promotion"
else:
    selected_row = screen.sort_values(["target_gap", "calib_rank_score", "max_dd"], ascending=[True, True, True]).iloc[0].to_dict()
    decision = "diagnostic_keep_no_promotion_calibration_failed"
selected_cfg = OverlayConfig(selected_row["score_name"], float(selected_row["entry"]), float(selected_row["exit"]), float(selected_row["cap"]), float(selected_row["min_account_dd"]), float(selected_row["trade_band"]), bool(selected_row["release_mom"]))
selected_daily, selected_meta = apply_score_overlay(base_for_overlay, selected_cfg)
selected_metrics = add_slice_metrics(metrics(selected_daily, f"{EXPERIMENT}_selected"), selected_daily, f"{EXPERIMENT}_selected")
selected_daily_path = CARD / f"{EXPERIMENT}_selected_soxl_daily.csv"
selected_daily.to_csv(selected_daily_path, index=False)
selection_path = CARD / "selection.json"
selection = {
    "experiment": EXPERIMENT,
    "baseline_path": str(BASE_PATH.relative_to(ROOT)),
    "raw_baseline_path": str(RAW_BASE_PATH.relative_to(ROOT)),
    "market_panel_path": str(PANEL_PATH.relative_to(ROOT)),
    "base_max_drawdown_span": base_span,
    "baseline": base_metrics,
    "raw_baseline": raw_metrics,
    "score_components": score_components,
    "screen_rows": int(len(screen)),
    "target_pass_rows": int(screen["target_pass"].sum()),
    "calib_valid_rows": int(screen["calib_valid"].sum()),
    "selected_config": selected_meta,
    "selected": selected_metrics,
    "decision": decision,
    "artifacts": {"notebook": str((CARD / f"{EXPERIMENT}.ipynb").relative_to(ROOT)), "feature_difference_screen": str(feature_diff_path.relative_to(ROOT)), "score_screen": str(screen_path.relative_to(ROOT)), "selected_daily": str(selected_daily_path.relative_to(ROOT))},
}
selection_path.write_text(json.dumps(selection, indent=2, ensure_ascii=False) + "\n")

comparison = pd.DataFrame([
    {"row": "raw_baseline", **{k: raw_metrics[k] for k in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "test_2022_max_dd", "test_2023_2024_max_dd", "holdout_2024_2026_max_dd"]}},
    {"row": "downside_vol_base", **{k: base_metrics[k] for k in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "test_2022_max_dd", "test_2023_2024_max_dd", "holdout_2024_2026_max_dd"]}},
    {"row": "selected_crash_risk", **{k: selected_metrics[k] for k in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "test_2022_max_dd", "test_2023_2024_max_dd", "holdout_2024_2026_max_dd"]}},
])

readout_lines = [
    "## Final Readout",
    f"Decision: `{decision}`.",
    f"Selected label: `{selected_meta['label']}`.",
    f"Target rows passing full-window `relative_return >= 30pp` and `max_dd < 15%`: `{selection['target_pass_rows']}` / `{selection['screen_rows']}`.",
    f"Selected full-window metrics: return `{selected_metrics['return']:.2f}%`, Sharpe `{selected_metrics['sharpe']:.3f}`, MaxDD `{selected_metrics['max_dd']:.2f}%`, excess `{selected_metrics['relative_return']:.2f}pp`, avg position `{selected_metrics['avg_pos']:.3f}`, trades `{selected_metrics['trades']}`.",
    f"Baseline full-window metrics: return `{base_metrics['return']:.2f}%`, Sharpe `{base_metrics['sharpe']:.3f}`, MaxDD `{base_metrics['max_dd']:.2f}%`, excess `{base_metrics['relative_return']:.2f}pp`, avg position `{base_metrics['avg_pos']:.3f}`, trades `{base_metrics['trades']}`.",
    f"Calibration `2020-2023`: selected excess `{selected_metrics['calib_2020_2023_relative_return']:.2f}pp`, MaxDD `{selected_metrics['calib_2020_2023_max_dd']:.2f}%`; baseline excess `{base_metrics['calib_2020_2023_relative_return']:.2f}pp`, MaxDD `{base_metrics['calib_2020_2023_max_dd']:.2f}%`.",
    f"Holdout `2024-2026`: selected excess `{selected_metrics['holdout_2024_2026_relative_return']:.2f}pp`, MaxDD `{selected_metrics['holdout_2024_2026_max_dd']:.2f}%`; baseline excess `{base_metrics['holdout_2024_2026_relative_return']:.2f}pp`, MaxDD `{base_metrics['holdout_2024_2026_max_dd']:.2f}%`.",
    "Interpretation: the crash-risk score is useful only if it reduces the near-frontier drawdown without destroying the already-thin `+30pp` excess budget. If no target pass exists, this notebook blocks promotion and preserves the factor-difference evidence for the next mechanism rather than promoting another threshold-tuned cap.",
    "Saved artifacts:",
    f"- `{selection['artifacts']['feature_difference_screen']}`",
    f"- `{selection['artifacts']['score_screen']}`",
    f"- `{selection['artifacts']['selected_daily']}`",
    f"- `{selection_path.relative_to(ROOT)}`",
]

display(Markdown("\n\n".join(readout_lines)))
display(Markdown(f"## Bad-Window Feature Difference\n\nBase max drawdown window: `{base_span['peak_date']}` -> `{base_span['trough_date']}` (`{base_span['max_dd']:.2f}%`). Saved: `{feature_diff_path.relative_to(ROOT)}`"))
display(feature_diff.head(18))
display(Markdown(f"## Overlay Screen\n\nScreen rows: `{len(screen)}`. Saved: `{screen_path.relative_to(ROOT)}`"))
display(screen.head(20)[["label", "target_pass", "calib_valid", "return", "max_dd", "relative_return", "avg_pos", "trades", "risk_active_days", "cap_changed_days", "calib_2020_2023_relative_return", "calib_2020_2023_max_dd", "holdout_2024_2026_relative_return", "holdout_2024_2026_max_dd", "target_gap"]])
display(comparison)
print(json.dumps(selection, indent=2, ensure_ascii=False))


## Final Readout

Decision: `diagnostic_keep_no_promotion`.

Selected label: `panic_rebound_guarded_e0.70_x0.65_cap0.05_dd0.07_band0.04_rel0`.

Target rows passing full-window `relative_return >= 30pp` and `max_dd < 15%`: `0` / `8640`.

Selected full-window metrics: return `190.75%`, Sharpe `1.002`, MaxDD `15.48%`, excess `44.60pp`, avg position `0.164`, trades `243`.

Baseline full-window metrics: return `177.48%`, Sharpe `0.960`, MaxDD `16.28%`, excess `31.33pp`, avg position `0.163`, trades `401`.

Calibration `2020-2023`: selected excess `21.70pp`, MaxDD `15.48%`; baseline excess `15.00pp`, MaxDD `15.99%`.

Holdout `2024-2026`: selected excess `-9.37pp`, MaxDD `14.83%`; baseline excess `-10.88pp`, MaxDD `16.28%`.

Interpretation: the crash-risk score is useful only if it reduces the near-frontier drawdown without destroying the already-thin `+30pp` excess budget. If no target pass exists, this notebook blocks promotion and preserves the factor-difference evidence for the next mechanism rather than promoting another threshold-tuned cap.

Saved artifacts:

- `runtime/execution/card/risk_budget_crash_risk_factor_v1/feature_difference_screen.csv`

- `runtime/execution/card/risk_budget_crash_risk_factor_v1/crash_risk_score_screen.csv`

- `runtime/execution/card/risk_budget_crash_risk_factor_v1/risk_budget_crash_risk_factor_v1_selected_soxl_daily.csv`

- `runtime/execution/card/risk_budget_crash_risk_factor_v1/selection.json`

## Bad-Window Feature Difference

Base max drawdown window: `2024-07-10` -> `2025-04-08` (`16.28%`). Saved: `runtime/execution/card/risk_budget_crash_risk_factor_v1/feature_difference_screen.csv`

,feature,bad_mean,good_mean,bad_median,good_median,effect_size_bad_minus_good,bad_count,good_count,abs_effect
0,lag_vix_pct_252,0.666293,0.365893,0.734127,0.313492,1.089520,359,1147,1.089520
1,lag_vix_z_126,0.693741,-0.306733,0.532626,-0.546003,0.839796,359,1168,0.839796
2,lag_mom_63,-0.083280,0.253280,-0.192728,0.239012,-0.762581,359,1146,0.762581
3,lag_ma100_gap,-0.096217,0.115182,-0.153943,0.146491,-0.704458,359,1160,0.704458
4,lag_ma50_gap,-0.083769,0.066160,-0.094308,0.098345,-0.674960,359,1185,0.674960
5,lag_vvix_z_126,0.546270,-0.188748,0.343016,-0.349988,0.635959,359,1168,0.635959
6,lag_mom_21,-0.071223,0.091550,-0.081278,0.090379,-0.634961,359,1188,0.634961
7,lag_drawdown_63,-0.318974,-0.198404,-0.305643,-0.132817,-0.622944,359,1189,0.622944
8,lag_soxl_soxx_ratio_widen_21,-0.068556,0.038968,-0.065309,0.049134,-0.619284,359,1188,0.619284
9,lag_vix_term,-0.067625,-0.111913,-0.068340,-0.122283,0.599592,359,1209,0.599592


## Overlay Screen

Screen rows: `8640`. Saved: `runtime/execution/card/risk_budget_crash_risk_factor_v1/crash_risk_score_screen.csv`

,label,target_pass,calib_valid,return,max_dd,relative_return,avg_pos,trades,risk_active_days,cap_changed_days,calib_2020_2023_relative_return,calib_2020_2023_max_dd,holdout_2024_2026_relative_return,holdout_2024_2026_max_dd,target_gap
0,panic_rebound_guarded_e0.70_x0.65_cap0.05_dd0....,False,True,190.745831,15.484927,44.599109,0.163622,243,123,290,21.700500,15.484927,-9.368629,14.831609,4.849269
1,panic_rebound_guarded_e0.70_x0.65_cap0.05_dd0....,False,True,190.745831,15.484927,44.599109,0.163622,243,122,290,21.700500,15.484927,-9.368629,14.831609,4.849269
2,panic_rebound_guarded_e0.75_x0.70_cap0.02_dd0....,False,True,183.304062,15.484927,37.157339,0.160758,257,203,351,21.296975,15.484927,-13.154441,15.192828,4.849269
3,panic_rebound_guarded_e0.75_x0.70_cap0.02_dd0....,False,True,183.304062,15.484927,37.157339,0.160758,257,200,351,21.296975,15.484927,-13.154441,15.192828,4.849269
4,panic_rebound_guarded_e0.70_x0.65_cap0.02_dd0....,False,True,182.568690,15.484927,36.421967,0.161378,249,123,329,16.686725,15.484927,-9.489943,12.825454,4.849269
5,panic_rebound_guarded_e0.70_x0.65_cap0.02_dd0....,False,True,182.568690,15.484927,36.421967,0.161378,249,122,329,16.686725,15.484927,-9.489943,12.825454,4.849269
6,panic_rebound_guarded_e0.75_x0.70_cap0.02_dd0....,False,True,179.301121,15.484927,33.154398,0.162170,249,110,326,19.819058,15.484927,-14.114239,15.299661,4.849269
7,panic_rebound_guarded_e0.75_x0.70_cap0.02_dd0....,False,True,179.301121,15.484927,33.154398,0.162170,249,107,326,19.819058,15.484927,-14.114239,15.299661,4.849269
8,panic_rebound_guarded_e0.75_x0.70_cap0.02_dd0....,False,True,186.037288,15.484927,39.890565,0.160160,259,229,365,23.058838,15.484927,-13.154441,15.192828,4.849269
9,panic_rebound_guarded_e0.75_x0.70_cap0.02_dd0....,False,True,186.037288,15.484927,39.890565,0.160160,259,226,365,23.058838,15.484927,-13.154441,15.192828,4.849269


,row,return,sharpe,max_dd,relative_return,avg_pos,trades,test_2022_max_dd,test_2023_2024_max_dd,holdout_2024_2026_max_dd
0,raw_baseline,148.914384,0.838944,22.153538,2.767661,0.180393,1485,21.978182,13.799342,18.322237
1,downside_vol_base,177.479796,0.960142,16.277862,31.333073,0.163345,401,11.189810,15.994587,16.277862
2,selected_crash_risk,190.745831,1.002457,15.484927,44.599109,0.163622,243,11.143979,15.484927,14.831609


{
  "experiment": "risk_budget_crash_risk_factor_v1",
  "baseline_path": "runtime/execution/card/risk_budget_downside_vol_target_v1/risk_budget_downside_vol_target_v1_best_downside_vol_soxl_daily.csv",
  "raw_baseline_path": "runtime/model/card/risk_budget_usage_v1/risk_budget_usage_v1_actor_parity_soxl_daily.csv",
  "market_panel_path": "data/processed/panels/soxl_soxx_vix_full.csv",
  "base_max_drawdown_span": {
    "peak_date": "2024-07-10",
    "trough_date": "2025-04-08",
    "max_dd": 16.277861775582814,
    "peak_position": 0.2321796565777723,
    "trough_position": 0.0454361578928488,
    "peak_price": 68.58999633789062,
    "trough_price": 8.25
  },
  "baseline": {
    "label": "risk_budget_downside_vol_target_v1_best_downside_vol",
    "rows": 1569,
    "start": "2020-01-02",
    "end": "2026-03-31",
    "return": 177.4797955898995,
    "sharpe": 0.9601416188320794,
    "max_dd": 16.277861775582814,
    "buy_and_hold_return": 146.14672276868933,
    "relative_return": 31.3330